In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Optional
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

In [4]:
# Creating output schema

class OutputSchema(BaseModel):

    feedback: Annotated[str, Field(description = "Generate a detailed feedback for the given essay")]
    score: Annotated[int, Field(description = "Give score to the given essay out of 10",
                               ge = 0, le = 10)]

# Creating structured model
structured_gemini = model_gemini.with_structured_output(OutputSchema)

# `Checking the model`

In [5]:
essay = '''Role of India in Artificial Intelligence
Introduction

Artificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century, reshaping industries, governance, and societies worldwide. For India—a nation with a massive population, vibrant IT sector, and rapidly expanding digital infrastructure—the rise of AI presents both opportunities and challenges. The role of India in AI is multifaceted, encompassing research, innovation, policy-making, talent development, and global collaboration. As one of the fastest-growing economies, India is poised to be a major contributor to the global AI ecosystem.

India’s Position in the Global AI Landscape

India has positioned itself as a significant player in the AI race, not by competing purely on cutting-edge innovation like the United States or China, but by focusing on AI for social good and inclusive growth. With over 750 million internet users, abundant data, and a thriving start-up ecosystem, India is leveraging AI in critical sectors such as agriculture, healthcare, education, and governance. Reports suggest that AI could contribute $1 trillion to India’s economy by 2035, highlighting its potential as a growth driver.

Government Initiatives and Policies

The Indian government has recognized AI as a strategic technology. Some notable initiatives include:

NITI Aayog’s National Strategy for Artificial Intelligence (2018) – Framed around the concept of “AI for All,” this strategy emphasizes responsible AI adoption in healthcare, agriculture, education, smart mobility, and smart cities.

National AI Portal (2020) – A platform to promote AI-related research, knowledge sharing, and collaboration.

Digital India Mission – Creating the necessary digital infrastructure for widespread AI adoption.

IndiaAI Mission (2023) – A national program to establish AI computing infrastructure, datasets, and ethical frameworks.

These initiatives reflect India’s approach of using AI not just for economic competitiveness, but also for addressing socio-economic challenges.

AI Research and Innovation in India

India has emerged as a hub of AI research and development, driven by both academia and industry:

Academic Contributions: Prestigious institutions like IITs, IISc Bangalore, and IIITs are conducting advanced research in machine learning, computer vision, and natural language processing (NLP). Indian researchers are particularly active in multilingual AI, focusing on building models for the country’s 22 scheduled languages.

Industry and Startups: India hosts more than 3,000 AI startups, many of which focus on affordable healthcare diagnostics, precision agriculture, fintech, and logistics. Unicorns like Ola, Flipkart, and Zomato are also deploying AI for customer engagement and operational efficiency.

Big Tech Presence: Companies like Google, Microsoft, Amazon, and IBM have AI research labs in India, contributing to global innovation while tapping into local talent.

Applications of AI in India

AI’s role in India extends beyond industry growth—it is transforming governance and public services:

Healthcare – AI-driven tools are improving early disease detection, telemedicine, and drug discovery. Startups are using AI for diagnosing tuberculosis, cancer, and eye disorders.

Agriculture – AI models predict crop yields, detect pest infestations, and optimize irrigation, benefiting farmers with small landholdings.

Education – AI-powered personalized learning platforms are bridging gaps in rural education.

Smart Cities & Governance – AI is used in traffic management, surveillance, and digital identity systems like Aadhaar to improve service delivery.

Financial Services – AI is enabling fraud detection, credit scoring, and digital banking for financial inclusion.

Challenges for India in AI Development

Despite its progress, India faces several challenges in becoming a global AI leader:

Data Privacy and Ethics – Balancing AI innovation with citizens’ rights remains a challenge due to the lack of comprehensive data protection laws.

Skilled Workforce Gap – While India produces millions of engineers annually, there is a shortage of highly skilled AI researchers and practitioners.

Infrastructure Limitations – AI research requires high computing power and reliable digital infrastructure, which are not yet universally available.

Unequal Access – There is a risk of AI benefits being concentrated in urban areas, leaving rural regions behind.

India’s Future Role in Global AI

Looking ahead, India’s role in AI is likely to be characterized by inclusivity, affordability, and ethical innovation. Instead of competing head-on with the U.S. and China in military or surveillance AI, India is prioritizing AI for sustainable development. Its future role could involve:

Becoming a global hub for AI talent, given its large pool of engineers and data scientists.

Leading in multilingual and culturally adaptive AI, making technology more inclusive for diverse populations.

Driving frugal innovation—building cost-effective AI solutions for healthcare, agriculture, and education that can be exported to other developing nations.

Shaping global AI ethics and governance frameworks, especially around responsible use and privacy.

Conclusion

India’s journey in Artificial Intelligence is not just about technological advancement but also about ensuring equitable growth and social impact. With its emphasis on “AI for All,” India is striving to create a unique AI ecosystem that balances innovation with inclusivity. By investing in research, nurturing talent, and adopting responsible governance, India has the potential to emerge as a global leader—not by mimicking other nations’ AI models, but by crafting one that addresses the needs of its people and sets an example for the developing world.'''

In [6]:
essay2 = '''India in Artificial Intelligent

India is now going big in artificial intelligence but still not very big like USA or China. Many company in India want to do AI but sometime no good resources and no enough expert people. Government also saying AI is future, but policy is sometime confusing and not always working good.

Lot of startup in Bangalore, Hyderabad and also Delhi doing AI thing like chatbot, health care machine, education software. But many project not complete proper and some only copy from west. Still India student is very smart and many go to America to study AI then later make big company or work in Google or Microsoft.

Indian AI also used in farming, like machine look at crop and tell farmer about soil, but sometime farmer no have internet so it not help much. In health also some hospital use AI for scan x-ray and find disease, but only big hospital in city have this, village hospital no have.

Future India want to be global hub for AI, but for that need more training, more infrastructure and less brain drain. Many people say if India combine AI with huge population and data, it can be very strong, but must also care about job lost, because robot and AI maybe take job from normal people.

So, India AI is like young baby, still learning walk, but one day maybe run fast if everything work out.'''

In [7]:
prompt = f'''Evaluate the language quality of the below given essay, give the feedback and generate a score out of 10

{essay}'''

response = structured_gemini.invoke(prompt)

In [8]:
print(response)

feedback="The essay demonstrates excellent language quality, characterized by clarity, coherence, and a sophisticated vocabulary. The structure is logical and well-organized, with a clear introduction, well-developed body paragraphs, and a strong conclusion. Sentence structures are varied and engaging, contributing to the readability of the text. There are no noticeable grammatical errors or spelling mistakes. The tone is informative and objective, suitable for the topic. The essay effectively covers a wide range of aspects concerning India's role in AI, from government initiatives and research to applications and challenges, providing a comprehensive overview. The use of specific examples to support points (e.g., NITI Aayog, IITs, specific companies) enhances the credibility and depth of the content. Overall, it is a very well-written and insightful piece." score=9


# `It's working. Now let's move to our workflow`

In [9]:
import operator

In [10]:
# Creating State
class EvaluationState(TypedDict):

    essay: str
    language_feedback: str
    aod_feedback: str    # Analysis_of_depth
    cot_feedback: str    # Clarity_of_thought
    summarized_feedback: str

    scores: Annotated[list[int], operator.add]
    average_score: float

# `operator.add`: This is called Reducer.

# What is a `Reducer`?
See, here we are taking scores from all three LLMs. if we use simple `score: int`, on updating `score` each time, it will replace with new_score. But we want all the three scores. Here comes `Reducer` into play. It doesn't replace the old score but it adds the new score as well. Hence, we will have all the three scores generated by all three LLMs.

For example: Here, our scores will be [8, 7, 6]

# How it works?

To add an element into the list, we use +. <br>
[5] + [6] + [7] results in [5, 6, 7]. <br>
but here, we cannot use `+`, hence we used `operator.add`

There are many other reducer functions like `min`, `max` etc. We will learn as we require.

In [11]:
['Nikunj'] + ['Jugal'] + [26] + [16]

['Nikunj', 'Jugal', 26, 16]

In [12]:
# Creating nodes

def language_analysis(state: EvaluationState) ->EvaluationState:

    # Fetchinh essay
    essay = state['essay']

    # Creating prompt without PromptTemplate
    prompt = f'''You are an expert writer, journalist and editor. Evaluate the below given essay
        based on language, provide a feedback and assign a score out of 10.
        
        {essay}'''
    
    # Invoking LLM
    response = structured_gemini.invoke(prompt)

    # Returning required state
    ret_state = {"language_feedback": response.feedback, "scores": [response.score]}

    return ret_state

#--------------------------------------------------------------------------------------------------------

def depth_analysis(state: EvaluationState) ->EvaluationState:

    # Fetchinh essay
    essay = state['essay']

    # Creating prompt without PromptTemplate
    prompt = f'''You are an expert writer, journalist and editor.
    Analyse depth of below essay, generate feedback and give score out of 10.
    
    {essay}'''
    
    # Invoking LLM
    response = structured_gemini.invoke(prompt)

    # Returning required state
    ret_state = {"aod_feedback": response.feedback, "scores": [response.score]}

    return ret_state

#--------------------------------------------------------------------------------------------------------

def cot_analysis(state: EvaluationState) ->EvaluationState:
    
    # Fetchinh essay
    essay = state['essay']

    # Creating PromptTemplate
    template = PromptTemplate(
        template = '''You are an expert writer, journalist and editor. Evaluate the below given essay
        based on clarity of thoughts, provide a feedback and assign a score out of 10.
        
        {essay}''',
        input_variables = ["essay"] 
    )

    # Invoking template
    prompt = template.invoke({"essay": essay})

    # Invoking LLM
    response = structured_gemini.invoke(prompt)

    # Returning required state
    ret_state = {"cot_feedback": response.feedback, "scores": [response.score]}

    return ret_state

#--------------------------------------------------------------------------------------------------------

def final_evaluation(state: EvaluationState) ->EvaluationState:

    # Fetching feedbacks and scores
    cot_feedback = state['cot_feedback']
    aod_feedback = state['aod_feedback']
    language_feedback = state['language_feedback']
    scores = state['scores']

    # Creating prompt without PromptTemplate
    prompt = f'''You are an expert writer, journalist and editor. Summarize the below given feedbacks.

    Clarity of Thought feedback - {cot_feedback}
    Analysis of Depth feedback - {aod_feedback}
    Language feedback - {language_feedback}
    '''
    
    # Invoking LLM
    response = model_gemini.invoke(prompt).content

    # Averaging the scores
    average_score = sum(scores)/len(scores)

    # Returning required state
    ret_state = {"summarized_feedback": response, "average_score": average_score}

    return ret_state

In [13]:
# Creating a graoh
graph = StateGraph(EvaluationState)

# Adding nodes
graph.add_node("lang_analysis", language_analysis)
graph.add_node("depth_analysis", depth_analysis)
graph.add_node("cot_analysis", cot_analysis)
graph.add_node('final_evaluation', final_evaluation)

# Creating Edges
graph.add_edge(START, 'lang_analysis')
graph.add_edge(START, 'depth_analysis')
graph.add_edge(START, 'cot_analysis')

graph.add_edge('cot_analysis', 'final_evaluation')
graph.add_edge('depth_analysis', 'final_evaluation')
graph.add_edge('lang_analysis', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [14]:
initial_state = {"essay": essay2}
response = workflow.invoke(initial_state)

In [15]:
response.keys()

dict_keys(['essay', 'language_feedback', 'aod_feedback', 'cot_feedback', 'summarized_feedback', 'scores', 'average_score'])

In [16]:
print(response["language_feedback"])
print("----------------------------------------------------------------------------------------------------------------------------")

print(response["aod_feedback"])
print("----------------------------------------------------------------------------------------------------------------------------")

print(response["cot_feedback"])
print("----------------------------------------------------------------------------------------------------------------------------")

print(response["summarized_feedback"])
print("----------------------------------------------------------------------------------------------------------------------------")

print(response["scores"])
print("----------------------------------------------------------------------------------------------------------------------------")

print(response["average_score"])

The essay provides a general overview of India's position in Artificial Intelligence, touching upon its challenges, initiatives, and future potential. However, the language used significantly detracts from its overall quality and readability. There are numerous grammatical errors, including issues with subject-verb agreement, article usage, and verb tenses (e.g., 'Many company in India want to do AI' instead of 'Many companies in India want to do AI'; 'policy is sometime confusing' instead of 'policies are sometimes confusing'). The sentence structure is often simplistic and repetitive, leading to a choppy flow. Vocabulary is basic and lacks the precision expected in an essay (e.g., 'doing AI thing' instead of 'developing AI applications'). The tone is overly informal, and punctuation is inconsistent. To improve, focus on strengthening foundational grammar, expanding vocabulary, varying sentence structures, and maintaining a formal academic tone. While the ideas are present, the execut

In [17]:
from rich.console import Console
from rich.panel import Panel

console = Console()

for key, value in response.items():
    console.print(Panel.fit(str(value), title=key.title(), border_style="purple"))


╭───────────────────────────────────────────────────── Essay ─────────────────────────────────────────────────────╮
│ India in Artificial Intelligent                                                                                 │
│                                                                                                                 │
│ India is now going big in artificial intelligence but still not very big like USA or China. Many company in     │
│ India want to do AI but sometime no good resources and no enough expert people. Government also saying AI is    │
│ future, but policy is sometime confusing and not always working good.                                           │
│                                                                                                                 │
│ Lot of startup in Bangalore, Hyderabad and also Delhi doing AI thing like chatbot, health care machine,         │
│ education software. But many project not complete proper and some only copy from west. Still India student is   │
│ very smart and many go to America to study AI then later make big company or work in Google or Microsoft.       │
│                                                                                                                 │
│ Indian AI also used in farming, like machine look at crop and tell farmer about soil, but sometime farmer no    │
│ have internet so it not help much. In health also some hospital use AI for scan x-ray and find disease, but     │
│ only big hospital in city have this, village hospital no have.                                                  │
│                                                                                                                 │
│ Future India want to be global hub for AI, but for that need more training, more infrastructure and less brain  │
│ drain. Many people say if India combine AI with huge population and data, it can be very strong, but must also  │
│ care about job lost, because robot and AI maybe take job from normal people.                                    │
│                                                                                                                 │
│ So, India AI is like young baby, still learning walk, but one day maybe run fast if everything work out.        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Language_Feedback ───────────────────────────────────────────────╮
│ The essay provides a general overview of India's position in Artificial Intelligence, touching upon its         │
│ challenges, initiatives, and future potential. However, the language used significantly detracts from its       │
│ overall quality and readability. There are numerous grammatical errors, including issues with subject-verb      │
│ agreement, article usage, and verb tenses (e.g., 'Many company in India want to do AI' instead of 'Many         │
│ companies in India want to do AI'; 'policy is sometime confusing' instead of 'policies are sometimes            │
│ confusing'). The sentence structure is often simplistic and repetitive, leading to a choppy flow. Vocabulary is │
│ basic and lacks the precision expected in an essay (e.g., 'doing AI thing' instead of 'developing AI            │
│ applications'). The tone is overly informal, and punctuation is inconsistent. To improve, focus on              │
│ strengthening foundational grammar, expanding vocabulary, varying sentence structures, and maintaining a formal │
│ academic tone. While the ideas are present, the execution of language needs substantial refinement to           │
│ effectively convey them.                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Aod_Feedback ──────────────────────────────────────────────────╮
│ The essay offers a foundational overview of Artificial Intelligence in India, touching upon several pertinent   │
│ areas such as government initiatives, the startup ecosystem, applications in sectors like healthcare and        │
│ agriculture, and future ambitions. It successfully identifies some critical challenges, including resource      │
│ limitations, policy ambiguity, and the issue of 'brain drain.' However, the essay is significantly hampered by  │
│ a profound lack of depth and detailed elaboration. Most points are presented as broad generalizations without   │
│ specific examples, data, or case studies to substantiate claims, rendering the analysis superficial. For        │
│ instance, assertions like 'policy is sometime confusing' or 'many project not complete proper' demand further   │
│ explanation and evidence. Furthermore, the essay is plagued by pervasive grammatical errors, awkward phrasing,  │
│ and incorrect word usage throughout (e.g., 'India in Artificial Intelligent,' 'Many company,' 'sometime,' 'no   │
│ good resources,' 'no enough expert people,' 'doing AI thing,' 'not complete proper,' 'farmer no have internet,' │
│ 'village hospital no have,' 'job lost'). The vocabulary employed is exceedingly basic, and the sentence         │
│ structures are simplistic, which severely undermines the essay's credibility and readability. To elevate the    │
│ quality of this essay, the author should prioritize: 1. Deepening the analysis by incorporating specific        │
│ examples, relevant data, or illustrative case studies to support all claims. 2. Rigorously proofreading and     │
│ correcting all grammatical errors, improving syntax, and ensuring correct word usage. 3. Expanding the          │
│ vocabulary to include more precise and sophisticated terms suitable for an analytical piece. 4. Developing a    │
│ clearer and more focused central thesis or argument to provide greater direction and impact. 5. Varying         │
│ sentence structures to enhance flow, readability, and overall sophistication. In summary, while the essay       │
│ provides a rudimentary introduction to the topic, it falls considerably short of the standards required for an  │
│ insightful, well-researched, or competently written analysis.                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Cot_Feedback ──────────────────────────────────────────────────╮
│ This essay attempts to provide an overview of India's position in Artificial Intelligence. While it touches     │
│ upon several relevant points such as India's growth in AI, challenges like resource scarcity and policy issues, │
│ startup activities, applications in farming and healthcare, and future aspirations, it suffers significantly    │
│ from grammatical errors, simplistic vocabulary, and informal sentence structures.                               │
│                                                                                                                 │
│ For instance, the title "India in Artificial Intelligent" should be "India in Artificial Intelligence". Phrases │
│ like "Many company", "no good resources", "sometime confusing", "Lot of startup", "AI thing", "not complete     │
│ proper", and "no have internet/hospital" are grammatically incorrect and indicate a lack of proficiency in      │
│ formal English writing. The essay reads more like a casual conversation than a formal piece of writing.         │
│                                                                                                                 │
│ To improve, focus on:                                                                                           │
│ 1.  **Grammar and Syntax:** Pay close attention to subject-verb agreement, tense consistency, article usage,    │
│ and proper sentence construction.                                                                               │
│ 2.  **Vocabulary:** Incorporate a wider range of academic and formal vocabulary to elevate the tone and         │
│ sophistication of the writing.                                                                                  │
│ 3.  **Clarity and Flow:** Use transition words and phrases to connect ideas smoothly between sentences and      │
│ paragraphs, ensuring a cohesive flow.                                                                           │
│ 4.  **Depth of Analysis:** While you've identified key areas, try to elaborate on them with specific examples   │
│ or more detailed explanations. For instance, when discussing "confusing policies," a brief example or context   │
│ would strengthen your point.                                                                                    │
│ 5.  **Formality:** Maintain a formal and objective tone appropriate for an essay.                               │
│                                                                                                                 │
│ The ideas presented are a good starting point, but the execution needs substantial improvement in terms of      │
│ language and academic rigor.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Summarized_Feedback ──────────────────────────────────────────────╮
│ As an expert editor, journalist, and writer, here is a consolidated summary of the feedback provided for the    │
│ essay on India's position in Artificial Intelligence:                                                           │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Editorial Summary: Assessment of "India in Artificial Intelligent"**                                          │
│                                                                                                                 │
│ **Overall Assessment:**                                                                                         │
│ The essay attempts to provide an overview of India's standing in Artificial Intelligence, touching upon         │
│ relevant topics such as growth, challenges (resource scarcity, policy issues, brain drain), startup activities, │
│ and applications in sectors like healthcare and agriculture. While the *ideas* presented offer a foundational   │
│ starting point and demonstrate an awareness of key areas, the **execution is severely hampered by pervasive and │
│ fundamental deficiencies in language, analytical depth, and academic rigor.** The essay, in its current form,   │
│ reads more like a casual conversation than a formal analytical piece.                                           │
│                                                                                                                 │
│ **Key Areas for Improvement:**                                                                                  │
│                                                                                                                 │
│ 1.  **Language and Mechanics (Most Critical Flaw):**                                                            │
│     *   **Grammar and Syntax:** The most significant issue is the widespread grammatical errors, including      │
│ incorrect subject-verb agreement, article usage, tense consistency, and basic sentence construction. Phrases    │
│ like "India in Artificial Intelligent," "Many company," "no good resources," "sometime confusing," "AI thing,"  │
│ and "not complete proper" are indicative of a profound lack of proficiency in formal English.                   │
│     *   **Vocabulary:** The vocabulary is consistently simplistic and lacks the precision, sophistication, and  │
│ academic tone expected in an analytical essay.                                                                  │
│     *   **Sentence Structure:** Sentence construction is often basic, repetitive, and awkward, leading to a     │
│ choppy flow and significantly detracting from readability.                                                      │
│     *   **Formality and Tone:** The essay maintains an overly informal, conversational tone, which is entirely  │
│ inappropriate for a piece of academic or journalistic writing. Punctuation is also inconsistent.                │
│                                                                                                                 │
│ 2.  **Depth of Analysis:**                                                                                      │
│     *   The essay suffers from a profound lack of depth and detailed elaboration. Points are presented as broad │
│ generalizations without sufficient substantiation through specific examples, relevant data, case studies, or    │
│ detailed explanations. Assertions such as "policy is sometime confusing" or "many project not complete proper"  │
│ demand further evidence and context to be credible and

╭─ Scores ──╮
│ [4, 3, 3] │
╰───────────╯

╭── Average_Score ───╮
│ 3.3333333333333335 │
╰────────────────────╯

# Note: While updating scores, we everywhere wrote `[response.score]`  (into a list)
`ret_state = {"cot_feedback": response.feedback, "scores": [response.score]}`  

Because we want to add new score to the list. Remember, whole creating schema, we initialized score = operator.add <br>
So, here, it will add each list of score to the existing scores' list.

scores = []

lang_analysis score = [8] <br>
scores + lang_analysis_score = [8] <br>

depth_score = [5]  <br>
scores + depth_score = [8, 5]

cot_score = [7]  <br>
scores + cot_score = [8, 5, 7]

# Sesssion complete.
# In our next session, we will learn `Conditional Workflow`.

# ***Thankyou***